In [1]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

def call_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

In [3]:
email = {
    "subject": "Can't login ",
    "body"   : "Our entire team can't login.  paid for annual "
               "plan last week. Board demo in 3 hours.entire team cant login",
    "correct": "Billing"
}

subject = email['subject']
body= email['body']

PROMPT_CURRENT = f"""You are an expert support email classifier.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Body    : {body}

Return ONLY: Category | Confidence (1-10)"""

result = call_llm(PROMPT_CURRENT)
result


'Technical | 8'

In [4]:
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Body    : {body}

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""


In [ ]:
result = call_llm(PROMPT_COT_V2)


"THINKING:  \n1. Surface complaint: The entire team can't log in.  \n2. Root cause: It seems there might be an account activation or access issue after an annual payment was made.  \n3. Team that OWNS this: Billing team (since they fix payment and account activation issues).  \n4. Business impact: High, as there is an upcoming board demo in 3 hours, and lack of access could negatively affect the presentation and professional relationship.\n\nCATEGORY: Billing"

In [6]:
print(result)

THINKING:  
1. Surface complaint: The entire team can't log in.  
2. Root cause: It seems there might be an account activation or access issue after an annual payment was made.  
3. Team that OWNS this: Billing team (since they fix payment and account activation issues).  
4. Business impact: High, as there is an upcoming board demo in 3 hours, and lack of access could negatively affect the presentation and professional relationship.

CATEGORY: Billing


In [ ]:
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

See the below examples
email : "I made the payment but did not received the payment confirmation"
category : dummy_category
THINKING:
1. Surface complaint: "I made the payment but did not received the payment confirmation"
2. Root cause: "Payment processing issue"
3. Team that OWNS this: "Billing"
4. Business impact: "Customer cannot access paid features"

email : "I made the payment but did not received the payment confirmation"
category : dummy_category

email : "I made the payment but did not received the payment confirmation"
category : dummy_category


Subject : {subject}
Body    : {body}

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""

result = call_llm(PROMPT_COT_V2)
print(result)


THINKING:
1. Surface complaint: The entire team is unable to log in.
2. Root cause: Possible account activation issue due to recent payment for the annual plan.
3. Team that OWNS this: Billing team.
4. Business impact: High, as it affects the entire team's access and there is an upcoming board demo in 3 hours.

CATEGORY: Billing


In [8]:
warmup_prompt = """You are a email support analyst. 
    
Before I give you a real email to classify, I want you to demonstrate 
your reasoning ability. Generate 3 different example emails a fintech 
support team might receive, and show step-by-step how you would 
classify each one. 

Cover different categories: payment failures, KYC issues, and fraud alerts.

Format each as:
EXAMPLE [N]:
Email: [email text you invent]
Reasoning: [step-by-step thinking]
Classification: [category + urgency]"""

self_examples = call_llm(warmup_prompt)

In [10]:
print(self_examples)

Certainly! Here are three example emails with reasoning and classification:

---

**EXAMPLE 1:**

**Email:**
Subject: Trouble with Payment Processing

Hi Fintech Support,

I tried to make a payment yesterday using your platform, but it failed. I checked my bank account and the funds are available. This is quite urgent as I need to complete this transaction today. Could you assist me in resolving this issue?

Thank you,
John Doe

**Reasoning:**
1. **Subject Analysis:** The subject mentions "Payment Processing," indicating a transaction-related issue.
2. **Content Details:** The customer describes a failed payment attempt despite having available funds, pointing to a payment failure rather than a lack of funds.
3. **Urgency Indicators:** The customer states “quite urgent” and emphasizes the need to complete the transaction “today.”
4. **Issue Identification:** The core issue is a payment failure, likely due to a technical glitch or a platform error rather than customer's financial status

In [11]:
email = {
    "subject": "Can't login ",
    "body"   : "Our entire team can't login.  paid for annual "
               "plan last week. Board demo in 3 hours.entire team cant login",
    "correct": "Billing"
}

subject = email['subject']
body= email['body']

In [12]:
classify_prompt = f"""You are a fintech support analyst.

You just demonstrated your reasoning ability with these examples:
{self_examples}

Now apply the SAME reasoning pattern to this real email:

# Subject: {subject}
Body: {body}

Follow the same format: Reasoning → Classification JSON."""

In [18]:
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Important:
- Root cause decides category, not surface words
- Pricing complaint → always Billing
- Feature requests alongside pricing → Billing wins
- 403 errors after payment failure → Billing not Technical

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""

results = []

for i in range(5):
    result = call_llm(PROMPT_COT_V2)
    for line in result.split("\n"):
        if line.startswith("CATEGORY:"):
            category = line.split("CATEGORY:")[1].strip()
            results.append(category)


In [19]:
from collections import Counter

category_counts = Counter(results)


winner = category_counts.most_common(1)[0]

for category, count in category_counts.items():
    bar = "##" * count
    print(f"  {category:20} : {bar} {count}/5")


print(f"Winner     : {winner[0]}")
print(f"Confidence : {winner[1]/5*100:.0f}%")

  Billing              : ########## 5/5
Winner     : Billing
Confidence : 100%
